In [ ]:
##################  3.4 Domain-specific patterns of moral convergence            #################     
##################  C3 Clustering of Moral Items Based on REAL--LLM Differences  #################     
 
import json
import textwrap

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform


# ============================================================
# 0. Function: pairwise RMS distance between items using pairwise deletion
# ============================================================

def pairwise_rms_item_distance_with_missing(X: pd.DataFrame) -> pd.DataFrame:
    """
    X: DataFrame with rows = countries, columns = items
    Returns an item × item distance matrix,
    computed using only countries observed for both variables.
    """
    item_names = X.columns.tolist()
    X_values = X.values.astype(float)

    n_items = X_values.shape[1]
    dist = np.zeros((n_items, n_items), dtype=float)

    for i in range(n_items):
        for j in range(i + 1, n_items):
            xi = X_values[:, i]
            xj = X_values[:, j]

            mask = ~np.isnan(xi) & ~np.isnan(xj)

            if mask.sum() == 0:
                d = np.nan
            else:
                diff = xi[mask] - xj[mask]
                d = np.sqrt(np.mean(diff ** 2))

            dist[i, j] = d
            dist[j, i] = d

    np.fill_diagonal(dist, 0.0)

    return pd.DataFrame(dist, index=item_names, columns=item_names)


# ============================================================
# 1. Load data
# ============================================================

df = pd.read_excel("country_dataset.xlsx")

df = df[df["Country_match"].notna()]
df = df[df["Country_match"].astype(str).str.strip() != ""]
df = df.copy()
df["Country_match"] = df["Country_match"].astype(str)
df = df.set_index("Country_match")

real_cols = [c for c in df.columns if c.startswith("REAL")]
llm_cols = [c for c in df.columns if c.startswith("LLM")]

def extract_trait(col):
    if "::" in col:
        return col.split("::", 1)[1].strip()
    return col.replace("REAL", "").replace("LLM", "").strip()

real_traits = {extract_trait(c): c for c in real_cols}
llm_traits = {extract_trait(c): c for c in llm_cols}

common_traits = sorted(set(real_traits) & set(llm_traits))
print("Number of items with both REAL and LLM:", len(common_traits))

if len(common_traits) == 0:
    raise ValueError("No common items found between REAL and LLM columns.")


# ============================================================
# 2. Build country × item DIFF matrix
# ============================================================

diff_data = pd.DataFrame(index=df.index)

for trait in common_traits:
    diff_data[trait] = (
        df[real_traits[trait]].astype(float)
        - df[llm_traits[trait]].astype(float)
    )


# ============================================================
# 3. Item standardization (without imputing missing values)
# ============================================================

means = diff_data.mean(axis=0, skipna=True)
stds = diff_data.std(axis=0, ddof=1, skipna=True).replace(0, np.nan)

diff_z = diff_data.sub(means, axis=1).div(stds, axis=1)


# ============================================================
# 4. Item clustering using pairwise deletion
# ============================================================

dist_items = pairwise_rms_item_distance_with_missing(diff_z)

max_dist = np.nanmax(dist_items.values)
dist_for_clust = dist_items.fillna(max_dist)

Z = linkage(squareform(dist_for_clust.values), method="average")

cut_height = 1.05

plt.figure(figsize=(12, 4))

ddata = dendrogram(
    Z,
    labels=dist_items.index.tolist(),
    leaf_rotation=0,
    leaf_font_size=8,
    color_threshold=cut_height,
    above_threshold_color="black"
)

wrapped = [textwrap.fill(lbl, 20) for lbl in ddata["ivl"]]

plt.xticks(
    ticks=plt.xticks()[0],
    labels=wrapped,
    rotation=45,
    ha="right",
    fontsize=8
)

plt.axhline(y=cut_height, color="grey", linestyle="--")
plt.title("Item clustering based on country-level REAL–LLM differences")

plt.tight_layout()
plt.savefig(
    "dendrogram_items_country_level_pairwise_missing.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


# ============================================================
# 5. Assign clusters to items
# ============================================================

clusters = fcluster(Z, t=cut_height, criterion="distance")

cluster_assign = pd.DataFrame({
    "Item": dist_items.index,
    "Cluster": clusters
})

cluster_dict = {
    c: cluster_assign.loc[cluster_assign["Cluster"] == c, "Item"].tolist()
    for c in sorted(cluster_assign["Cluster"].unique())
}

print("\nCluster items:")
for c, v in cluster_dict.items():
    print(f"{c}: {v}")


# ============================================================
# 6. Item cluster labels
# ============================================================

cluster_label_map = {
    1: "Personal and bioethical morality",
    2: "Public / Social Order Morality",
    3: "Anti-civic norm",
    4: "Political Violence"
}

cluster_ids = sorted(cluster_dict.keys())


# ============================================================
# 7. Country group definition: WEIRD vs Rest of the World
# ============================================================

weird_countries = [
    'Andorra', 'Australia', 'Austria', 'Canada', 'Czechia', 'Denmark',
    'Finland', 'France', 'Germany', 'Great Britain', 'Iceland', 'Italy',
    'Japan', 'Netherlands', 'New Zealand', 'Northern Ireland', 'Norway',
    'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland',
    'United States of America', 'Uruguay'
]

groups = pd.Series(
    np.where(diff_data.index.isin(weird_countries), "WEIRD", "Rest of the World"),
    index=diff_data.index,
    name="Group"
)

group_names = ["WEIRD", "Rest of the World"]


# ============================================================
# 8. Static REAL vs LLM scatterplot by cluster
# ============================================================

colors = {
    "WEIRD": "#1f77b4",
    "Rest of the World": "#d62728"
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.flatten()

clusters_to_plot = cluster_ids[:4]

for ax, c in zip(axes, clusters_to_plot):
    vars_c = cluster_dict[c]

    real_cols_c = [real_traits[v] for v in vars_c if v in real_traits]
    llm_cols_c = [llm_traits[v] for v in vars_c if v in llm_traits]

    if len(real_cols_c) == 0 or len(llm_cols_c) == 0:
        ax.set_visible(False)
        continue

    real_mean = df[real_cols_c].mean(axis=1, skipna=True)
    llm_mean = df[llm_cols_c].mean(axis=1, skipna=True)

    for g in group_names:
        mask = groups == g

        x = real_mean[mask]
        y = llm_mean[mask]

        valid = x.notna() & y.notna()

        ax.scatter(
            x[valid],
            y[valid],
            color=colors[g],
            alpha=0.35,
            label=g,
            s=90
        )

    ax.plot([1, 10], [1, 10], linestyle="--", color="grey")
    ax.set_xlim(1, 10)
    ax.set_ylim(1, 10)
    ax.set_title(cluster_label_map.get(c, f"Cluster {c}"))
    ax.set_xlabel("WVS/EVS")
    ax.set_ylabel("LLM")

for ax in axes[len(clusters_to_plot):]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.98)
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(
    "scatter_clusters_WEIRD_vs_ROW_pairwise_missing.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()